In [ ]:

import os
import time
import pandas as pd
import sys

from datetime import datetime
import pytz

from data_loader import allocate_malicious_nodes
from data_loader import generate_topology
from main_cnn_GPU import run_simulation_CNN_GPU
from data_loader import set_seed
from data_loader import distribute_data
from data_loader import get_data
from theoretical_intensity import calculate_theoretical_intensity

import copy
from torch.utils.data import DataLoader


original_stdout = sys.stdout
original_stderr = sys.stderr
class DualLogger(object):
    def __init__(self, file_path):
        self.terminal = sys.stdout
        self.log = open(file_path, "a", encoding='utf-8')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

    def close(self):
        self.log.close()

import sys

NORM_FACTOR = 1
# ==========================================
# 1.Experiment settup
# ==========================================



NUM_CLIENTS = 20
BOOST_FACTORS = 2
MALICIOUS_RATIO = 0.3
defense_budget  = int(NUM_CLIENTS * 0.2)
GLOBAL_ROUNDS = 15
EPOCHS = 5
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0

SCALE_FACTOR = 0.8
train_ds, test_ds = get_data()
test_loader = DataLoader(test_ds, batch_size=256)


SEEDS = [1,2,3]
MAL_RATIOS = [ 0.3, 0.1, 0.2]
TOPO_TYPES = [ 'scale_free','random_regular']
DEFENSE_RATIOS = [ 0.2]
mech = 'MAB'
NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
NORM_FACTOR = 1
SCALE_FACTOR = 0.8
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0
def_ratio = 0.2
mal_ratio = 0.3
AUDIT_PROBS = [0.8, 0.9]
AGG_PROBS = [0.8, 0.9]
AGG_THRE = [0.4, 0.5]

ratio = 0.3
num_mal= int(NUM_CLIENTS*ratio)
current_def_budget = int(NUM_CLIENTS*def_ratio)
SAVE_PATH = ''
for topo_type in TOPO_TYPES:
    for seed_val in SEEDS:
        for aup in AUDIT_PROBS:
            for ap in AGG_PROBS:
                for at in AGG_THRE:

                    print(f"\n{'='*60}")
                    print(f"⏰ : {time.strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"📡 : Topo={topo_type}, Mal={mal_ratio}, Def={def_ratio}")
                    print(f"{'='*60}")
                    print(f"\n{'#'*60}")
                    print(f"📡 Topo: {topo_type} | Mal: {mal_ratio} | Def: {def_ratio} | Seed: {seed_val}")
                    print(f"{'#'*60}")

                    set_seed(seed_val)
                    client_datasets = distribute_data(train_ds, NUM_CLIENTS)
                    G = generate_topology(NUM_CLIENTS, topo_type)
                    neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}


                    malicious_clients, defense_nodes = allocate_malicious_nodes(
                        G, num_mal, current_def_budget, topology_type=topo_type,placement='Topology-Aware',# placement='Topology-Aware'
                    )

                    theo_intensities = calculate_theoretical_intensity(
                        neighbors, malicious_clients, NUM_CLIENTS, BOOST_FACTORS, lambda_benign=0.3
                    )
                  
                    param_str = f"AUP{aup}_AP{ap}_AT{at}"

                    csv_filename = f"Final_CNN_{topo_type}_MR{ratio}_DR{def_ratio}_{mech}_{param_str}_seed{seed_val}.csv"
                    log_filename = f"Final_CNN_{topo_type}_MR{ratio}_DR{def_ratio}_{mech}_{param_str}_seed{seed_val}.txt"

                    full_save_path = os.path.join(SAVE_PATH, csv_filename)
                    log_full_path = os.path.join(SAVE_PATH, log_filename)

                    logger = DualLogger(log_full_path)
                    sys.stdout = logger
                    sys.stderr = logger
                    shanghai_tz = pytz.timezone('Asia/Shanghai')
                    print(f"🚀 Running: {mech}")
                    start_wall_time = datetime.now(shanghai_tz).strftime("%Y-%m-%d %H:%M:%S")
                    start_tick = time.time()

                    _, _, accs, asrs = run_simulation_CNN_GPU(
                        seed_val, NUM_CLIENTS, defense_nodes, malicious_clients,
                        G, neighbors, client_datasets, test_loader,
                        atk_type=ATK_TYPE,
                        mechanism=mech,
                        intensity=INTENSITY,
                        norm_factor=NORM_FACTOR,
                        debug_mode=False,
                        GLOBAL_ROUNDS=GLOBAL_ROUNDS,
                        epochs=5, debug=False,audit_prob=aup,
                        agg_prob=ap,
                        agg_threshold=at
                    )
                    end_tick = time.time()
                    duration_sec = round(end_tick - start_tick, 2)

                    result_entry_base = {
                        'seed': seed_val,
                        'mechanism': mech,
                        'audit_prob': aup,
                        'agg_prob': ap,
                         'agg_thre': at,
                        'malicious_ratio': mal_ratio,
                        'defense_ratio': def_ratio,
                        'topology': topo_type,
                        'norm_factor': NORM_FACTOR,
                        'scale_factor': SCALE_FACTOR,
                        'global_rounds': GLOBAL_ROUNDS,
                        'start_time': start_wall_time,
                        'duration_sec': duration_sec
                    }
                    all_results = []
                    for i in range(NUM_CLIENTS):
                        client_row = copy.deepcopy(result_entry_base)
                        client_row.update({
                            'client_id': i,
                            'final_acc': accs[i],
                            'final_asr': asrs[i],
                            'theo_intensity': theo_intensities[i] if i < len(theo_intensities) else 0.0,
                            'node_type': 'MAL' if i in malicious_clients else ('DEF' if i in defense_nodes else 'BEN')
                        })
                        all_results.append(client_row)

                    pd.DataFrame(all_results).to_csv(full_save_path, index=False)


                    sys.stdout = original_stdout


                    sys.stdout = logger
print(f"\n🎉 Experiments compeleted！")

In [ ]:
#Table 4
import os
import glob
import pandas as pd



list_df = []
def fmt(val_mean, val_std):
    m = val_mean
    s = (0.0 if pd.isna(val_std) else val_std)
    return f"{m:05.2f} ({s:05.2f})"
SAVE_PATH = ''
files_sens = glob.glob(os.path.join(SAVE_PATH, "Final_CNN*.csv"))


for f in files_sens:
    temp_df = pd.read_csv(f)
    list_df.append(temp_df)

df_results = pd.concat(list_df, ignore_index=True)
df_benign = df_results[df_results['node_type'] != 'MAL'].copy()

analysis_df = df_benign.groupby(['audit_prob', 'agg_prob', 'agg_thre']).agg({
    'final_acc': 'mean',
    'final_asr': 'mean'
}).reset_index()

print(analysis_df)

# ==========================================
# 
# ==========================================
if list_df:

    trial_means = df_benign.groupby(
        ['topology', 'audit_prob', 'agg_prob', 'agg_thre', 'seed']
    )[['final_acc', 'final_asr']].mean().reset_index()

    stats = trial_means.groupby(
        ['topology', 'audit_prob', 'agg_prob', 'agg_thre']
    )[['final_acc', 'final_asr']].agg(['mean', 'std']).reset_index()

    stats.columns = [
        '_'.join(col).strip('_') if isinstance(col, tuple) else col
        for col in stats.columns.values
    ]

    def generate_latex_rows(df_subset):
        rows = []
        df_subset = df_subset.sort_values(['audit_prob', 'agg_prob', 'agg_thre'])
        for _, row in df_subset.iterrows():
            aup = f"{row['audit_prob']:.1f}"
            ap = f"{row['agg_prob']:.1f}"
            at = f"{row['agg_thre']:.1f}"
            acc = fmt(row['final_acc_mean'], row['final_acc_std'])
            asr = fmt(row['final_asr_mean'], row['final_asr_std'])
            rows.append(f"{aup} & {ap} & {at} & {acc} & {asr} \\\\")

       
            if at == "0.5":
                rows.append(r"\cmidrule(lr){2-5}")
        return "\n".join(rows)

    topologies = stats['topology'].unique()
    table_body = ""
    for topo in topologies:
        topo_label = "Scale-free" if topo == 'scale_free' else "Random-regular"
        subset = stats[stats['topology'] == topo]
        table_body += f"\n\\midrule\n\\multicolumn{{5}}{{l}}{{\\textit{{Topology: {topo_label}}}}} \\\\ \n\\midrule\n"
        table_body += generate_latex_rows(subset)

    latex_final = r"""
\begin{table}[htbp]
    \centering
    \caption{Sensitivity Analysis of MAB Hyperparameters(CNN on GTSRB)}
    \label{tab:mab_sensitivity_split}
    \small
    \begin{tabular}{ccccc}
        \toprule
        \textbf{ADR} & \textbf{AGR} &  $\tau_{agg}$ & \textbf{ACC (\%)} & \textbf{ASR (\%)} \\
        \midrule
""" + table_body + r"""
        \bottomrule
    \end{tabular}
\end{table}
"""

    latex_final = latex_final.replace(r"\cmidrule(lr){2-5}" + "\n\n\\midrule", r"\midrule")
    latex_final = latex_final.replace(r"\cmidrule(lr){2-5}" + "\n\n\\bottomrule", r"\bottomrule")

    print(latex_final)